In [34]:
%load_ext autoreload
%autoreload 2



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import timm

from pathlib import Path

MAIN = Path("..")

In [36]:
BASE_DIRS = {
    "Swin image-only": (
        MAIN / "outputs" / "extra" /
        "exp3_swin_384_strong_MSE_MLPHead"
    ),

    "Swin concat": (
        MAIN / "outputs" / "extra" /
        "exp5_SwinT_StrongAug_MLPHead_mse"
    ),

    "Swin Cross-Attention": (
        MAIN / "outputs" / "extra" /
        "exp13_swin_cross_attn_mse_loss_single_block_tab_transformer"
    ),
}
#paths of default seed 42 experiment and other seeds 50, 55
SEED_SUFFIXES = {
    42: "",
    50: "_Seed50",
    55: "_Seed55",
}

N_BOOT = 5000

In [37]:
# to check if the all models oof predictions are available 
rows = []

for model_name, base_dir in BASE_DIRS.items():
    for seed, suffix in SEED_SUFFIXES.items():

        oof_path = Path(str(base_dir) + suffix) / "oof_detail.csv"

        rows.append({
            "Model": model_name,
            "Seed": seed,
            "OOF available": oof_path.exists(),
            "OOF path": str(oof_path),
        })

oof_models_config_combined = pd.DataFrame(rows)

oof_models_config_combined

,Model,Seed,OOF available,OOF path
0,Swin image-only,42,True,../outputs/extra/exp3_swin_384_strong_MSE_MLPH...
1,Swin image-only,50,True,../outputs/extra/exp3_swin_384_strong_MSE_MLPH...
2,Swin image-only,55,True,../outputs/extra/exp3_swin_384_strong_MSE_MLPH...
3,Swin concat,42,True,../outputs/extra/exp5_SwinT_StrongAug_MLPHead_...
4,Swin concat,50,True,../outputs/extra/exp5_SwinT_StrongAug_MLPHead_...
5,Swin concat,55,True,../outputs/extra/exp5_SwinT_StrongAug_MLPHead_...
6,Swin Cross-Attention,42,True,../outputs/extra/exp13_swin_cross_attn_mse_los...
7,Swin Cross-Attention,50,True,../outputs/extra/exp13_swin_cross_attn_mse_los...
8,Swin Cross-Attention,55,True,../outputs/extra/exp13_swin_cross_attn_mse_los...


In [38]:
# get the path of the oof_detail.csv file for a given model and seed
def get_oof_path(model_name, seed):
    base_dir = BASE_DIRS[model_name]
    suffix = SEED_SUFFIXES[seed]

    path = Path(str(base_dir) + suffix) / "oof_detail.csv"

    if not path.exists():
        raise FileNotFoundError(f"OOF file not found:\n{path}")

    return path

# load the oof_detail.csv file for a given model and seed
def load_oof(path, model_name):
    df = pd.read_csv(path)

    pred_col = "oof_pred" if "oof_pred" in df.columns else "final_pred"

    if "Id" not in df.columns or "ytrue" not in df.columns:
        raise ValueError(f"Missing Id or ytrue column:\n{path}")

    if pred_col not in df.columns:
        raise ValueError(f"Prediction column not found:\n{path}")

    if df["Id"].duplicated().any():
        raise ValueError(f"Duplicated Id values found:\n{path}")

    return df[["Id", "ytrue", pred_col]].rename(
        columns={pred_col: model_name}
    )

# to create a paired dataframe of oof predictions for two models with the same Id and ytrue values
def make_paired_oof(baseline_name, candidate_name, seed):
    baseline = load_oof(
        get_oof_path(baseline_name, seed),
        baseline_name
    )

    candidate = load_oof(
        get_oof_path(candidate_name, seed),
        candidate_name
    )

    paired = baseline.merge(
        candidate,
        on="Id",
        how="inner",
        suffixes=("_baseline", "_candidate"),
        validate="one_to_one",
    )

    if len(paired) == 0:
        raise ValueError("No matching Id values found.")

    if not np.allclose(
        paired["ytrue_baseline"],
        paired["ytrue_candidate"]
    ):
        raise ValueError("True target values do not match.")

    paired = paired.rename(
        columns={"ytrue_baseline": "ytrue"}
    )

    paired = paired.drop(columns=["ytrue_candidate"])

    return paired.sort_values("Id").reset_index(drop=True)

#### RMSE Diffence Confidence interval
We have out-of-fold predictions for each model on the same ~9,912 samples. To compare two models fairly, we randomly resample these samples with replacement, using the same resampled samples for both models in each bootstrap iteration so that both models are evaluated on identical observations. For each resampled set, we calculate the RMSE for each model and compute their difference, defined as ΔRMSE=RMSE(modelB)-RMSE(modelA)
. We repeat this process 5,000 times, producing a bootstrap distribution of RMSE differences that reflects the variability in the estimated performance difference due to resampling. From this distribution, we obtain a 95% bootstrap confidence interval using the 2.5th and 97.5th percentiles.

The interpretation depends primarily on whether the confidence interval contains zero. If the interval contains zero, there is no clear evidence of a performance difference between the two models. If the interval is entirely negative, Model B has lower RMSE than Model A, indicating better performance by Model B. If the interval is entirely positive, Model B has higher RMSE than Model A, indicating better performance by Model A. The magnitude of the difference should also be considered: values closer to zero indicate a smaller performance difference, whereas values farther from zero indicate a larger difference.

In [39]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def paired_bootstrap(
    y_true,
    baseline_pred,
    candidate_pred,
    n_boot=5000,
    seed=42,
    batch_size=100,
):
    y_true = np.asarray(y_true, dtype=np.float64)
    baseline_pred = np.asarray(baseline_pred, dtype=np.float64)
    candidate_pred = np.asarray(candidate_pred, dtype=np.float64)

    n = len(y_true)

    baseline_rmse = rmse(y_true, baseline_pred)
    candidate_rmse = rmse(y_true, candidate_pred)

    observed_reduction = baseline_rmse - candidate_rmse

    rng = np.random.default_rng(seed)
    reductions = np.empty(n_boot)

    start = 0

    while start < n_boot:
        current_batch = min(batch_size, n_boot - start)

        # The same sampled OOF indices are used for both models.
        idx = rng.integers(0, n, size=(current_batch, n))

        y_boot = y_true[idx]

        baseline_boot_rmse = np.sqrt(
            np.mean(
                (y_boot - baseline_pred[idx]) ** 2,
                axis=1
            )
        )

        candidate_boot_rmse = np.sqrt(
            np.mean(
                (y_boot - candidate_pred[idx]) ** 2,
                axis=1
            )
        )

        reductions[start:start + current_batch] = (
            baseline_boot_rmse - candidate_boot_rmse
        )

        start += current_batch

    ci_low, ci_high = np.quantile(reductions, [0.025, 0.975])

    return {
        "baseline_rmse": baseline_rmse,
        "candidate_rmse": candidate_rmse,
        "rmse_reduction": observed_reduction,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "reductions": reductions,
    }

In [50]:
COMPARISONS = [
    ("Swin image-only", "Swin concat"),
    ("Swin image-only", "Swin Cross-Attention"),
    ("Swin concat", "Swin Cross-Attention"),
]

results_rows = []
bootstrap_results = {}

for baseline_name, candidate_name in COMPARISONS:
    for seed in SEED_SUFFIXES:

        try:
            paired_df = make_paired_oof(
                baseline_name,
                candidate_name,
                seed
            )

            result = paired_bootstrap(
                y_true=paired_df["ytrue"].values,
                baseline_pred=paired_df[baseline_name].values,
                candidate_pred=paired_df[candidate_name].values,
                n_boot=N_BOOT,
                seed=seed,
            )

           

            results_rows.append({
                "Seed": seed,
                "Baseline": baseline_name,
                "Candidate": candidate_name,
                "Baseline RMSE": result["baseline_rmse"],
                "Candidate RMSE": result["candidate_rmse"],
                "RMSE Reduction": result["rmse_reduction"],
                "95% CI lower": result["ci_low"],
                "95% CI upper": result["ci_high"]
            })

            bootstrap_results[
                (baseline_name, candidate_name, seed)
            ] = result["reductions"]

        except FileNotFoundError as error:
            print(error)

paired_results = pd.DataFrame(results_rows)

display(
    paired_results.sort_values(
        ["Baseline", "Candidate", "Seed"]
    ).reset_index(drop=True)
)

,Seed,Baseline,Candidate,Baseline RMSE,Candidate RMSE,RMSE Reduction,95% CI lower,95% CI upper
0,42,Swin concat,Swin Cross-Attention,17.833680,17.593708,0.239973,0.122570,0.361074
1,50,Swin concat,Swin Cross-Attention,17.799803,17.723151,0.076652,-0.051905,0.198239
2,55,Swin concat,Swin Cross-Attention,17.776563,17.814603,-0.038039,-0.154941,0.081276
3,42,Swin image-only,Swin Cross-Attention,17.736021,17.593708,0.142313,0.032857,0.255065
4,50,Swin image-only,Swin Cross-Attention,17.762967,17.723151,0.039816,-0.085128,0.166987
5,55,Swin image-only,Swin Cross-Attention,17.742166,17.814603,-0.072437,-0.190582,0.052742
6,42,Swin image-only,Swin concat,17.736021,17.833680,-0.097660,-0.190224,-0.007282
7,50,Swin image-only,Swin concat,17.762967,17.799803,-0.036836,-0.133799,0.060293
8,55,Swin image-only,Swin concat,17.742166,17.776563,-0.034398,-0.128881,0.056743


## Paramter Count

In [ ]:
from src.models import VisionRegNet
# ============================================================
# The two models which were used for vision backbone change effect analysis
# both of these were trained with the same configuration including batch size 12, loss BCE,
# ============================================================

BACKBONE_CONFIGS = {
    "EfficientNet-B1 image-only": {
        "backbone": "efficientnet_b1",
        "img_size": 384,
        "batch_size": 12,
        "loss": "BCE",
        "aug":"strong",
        "head_hidden": 256,
    },

    "Swin image-only": {
        "backbone": "swin_large_patch4_window12_384",
        "img_size": 384,
        "batch_size": 12,
        "loss": "BCE",
        "aug":"strong",
        "head_hidden": 256,
    },
}


def count_parameters(model, trainable_only=False):
    if trainable_only:
        return sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )

    return sum(
        p.numel()
        for p in model.parameters()
    )


rows = []

for model_name, cfg in BACKBONE_CONFIGS.items():

    backbone_name = cfg["backbone"]
    img_size = cfg["img_size"]

    extra_kwargs = {}

    if "swin" in backbone_name.lower() or "vit" in backbone_name.lower():
        extra_kwargs["img_size"] = img_size
        extra_kwargs["dynamic_img_pad"] = True

    # pretrained=False only avoids downloading weights.
    # It does not change dimensions or parameter counts.
    backbone = timm.create_model(
        backbone_name,
        pretrained=False,
        num_classes=0,
        **extra_kwargs,
    )

    full_model = VisionRegNet(
        backbone_name=backbone_name,
        img_size=img_size,
        head_hidden=cfg["head_hidden"],
        pretrained=False,
    )

    pretrained_cfg = getattr(backbone, "pretrained_cfg", {}) or {}

    pretrained_id = (
        pretrained_cfg.get("hf_hub_id")
        or pretrained_cfg.get("url")
        or pretrained_cfg.get("architecture")
        or "Check installed timm pretrained_cfg"
    )

    rows.append({
        "Model": model_name,
        "Exact backbone": backbone_name,
        "Pretrained configuration": pretrained_id,
        "Feature dimension": backbone.num_features,
        "Backbone parameters": count_parameters(backbone),
        "Backbone parameters (M)": round(
            count_parameters(backbone) / 1_000_000,
            3
        ),
        "Full image-only parameters": count_parameters(full_model),
        "Full image-only parameters (M)": round(
            count_parameters(full_model) / 1_000_000,
            3
        ),
        "Input resolution": f"{img_size} x {img_size}",
        "Batch size": cfg["batch_size"],
        "Loss": cfg["loss"],
    })

capacity_table = pd.DataFrame(rows)



In [46]:
for _, row in capacity_table.iterrows():

    print("=" * 75)
    print("Model:", row["Model"])
    print("Exact backbone:", row["Exact backbone"])
    print("Pretrained configuration:", row["Pretrained configuration"])
    print("Feature dimension:", row["Feature dimension"])
    print(
        "Backbone parameters:",
        f"{row['Backbone parameters']:,}",
        f"({row['Backbone parameters (M)']:.3f} M)"
    )
    print(
        "Full image-only parameters:",
        f"{row['Full image-only parameters']:,}",
        f"({row['Full image-only parameters (M)']:.3f} M)"
    )
    print("Input resolution:", row["Input resolution"])
    print("Batch size:", row["Batch size"])
    print("Loss:", row["Loss"])

Model: EfficientNet-B1 image-only
Exact backbone: efficientnet_b1
Pretrained configuration: timm/efficientnet_b1.ra4_e3600_r240_in1k
Feature dimension: 1280
Backbone parameters: 6,513,184 (6.513 M)
Full image-only parameters: 6,514,465 (6.514 M)
Input resolution: 384 x 384
Batch size: 12
Loss: BCE
Model: Swin image-only
Exact backbone: swin_large_patch4_window12_384
Pretrained configuration: timm/swin_large_patch4_window12_384.ms_in22k_ft_in1k
Feature dimension: 1536
Backbone parameters: 195,198,516 (195.199 M)
Full image-only parameters: 195,200,053 (195.200 M)
Input resolution: 384 x 384
Batch size: 12
Loss: BCE
